In [1]:
from spin_lattices import KagomeLattice, SpinLattice, ChainLattice, SquareLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import networkx as nx
import numpy as np
from typing import Callable
import torch
import numpy.typing as npt
import lattice_symmetries as ls
from typing import Any, Optional, Union, Dict, Tuple
from loguru import logger
from collections import namedtuple
from torch import Tensor
import torch.nn as nn
from utils import make_unpacked_configurations
import io
from contextlib import redirect_stderr
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

2023-07-10 17:50:45.913 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-07-10 17:50:45.917 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-07-10 17:50:45.969 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


In [18]:
lattice = ChainLattice(10)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=False,
    spin_inversion=None,
)
system.get_eigenstates(1);
with redirect_stderr(io.StringIO()) as f:
    system.hamiltonian.apply_diag_to_basis_state(system.basis.states[0])

2023-07-10 17:52:58.149 | DEBUG    | heisenberg_hamiltonians:__init__:456 - number_spins=10
2023-07-10 17:52:58.150 | DEBUG    | heisenberg_hamiltonians:__init__:466 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-10 17:52:58.157 | DEBUG    | heisenberg_hamiltonians:__init__:475 - Hilbert space dimension is 252
2023-07-10 17:52:58.160 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:64 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-ChainLattice10x1-1.0-1.0-False-None-1.pickle
2023-07-10 17:52:58.161 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:111 - Ground state energy is -18.0617854180
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...


In [3]:
def get_relsigns(
    system: SpinSystem, cluster: npt.NDArray[np.uint64], log_prob_fn: Callable[[Tensor], Tensor]
) -> npt.NDArray[np.uint64]:
    return np.sign(system.get_ground_state_coeffs(cluster)) * np.random.choice([-1, 1])

def overlap(x, y):
    x = x.view(-1)
    y = y.view(-1)
    return torch.sum(x * y) / torch.sqrt(torch.sum(x**2) * torch.sum(y**2))

In [5]:
# def energy_loc(
#     hamiltonian: ls.Operator,
#     sigmas: torch.Tensor,
#     log_amplitude: Callable,
#     cluster: torch.Tensor,
#     relsigns: torch.Tensor,
# ):
#     """Local energy of a spin configuration.

#     Args:
#         hamiltonian: Hamiltonian.
#         sigma: Spin configuration.
#         log_amplitude: Function that calculates logarithm of
#                        the amplitude of wave function.
#         cluster: Cluster of states, must contain sigma.
#         relsigns: Relative signs of the cluster.

#     Returns:
#         Local energy.
#     """
#     cluster = torch.sort(cluster)
#     n_sigmas = len(sigmas)

#     coeffs = {}
#     neighbours = {}

#     for sigma in sigmas:
#         coeffs_, neighbours_ = zip(*hamiltonian.apply_off_diag_to_basis_state(int(sigma.item())))
#         coeffs[sigma] = torch.tensor(coeffs_)
#         neighbours[sigma] = torch.tensor(neighbours_)

#     all_neighbours = torch.sort(torch.unique(torch.concatenate(list(neighbours.values()))))

#     sigmas_idx = torch.searchsorted(cluster, sigmas)
#     neighbours_idx = torch.searchsorted(cluster, all_neighbours)

#     coeffs = torch.ones((n_sigmas, len(all_neighbours))) * (-10000)
#     for i, sigma in enumerate(sigmas):
#         for neighbour, coeff in zip(neighbours[sigma], coeffs[sigma]):
#             coeffs[i, torch.searchsorted(all_neighbours, neighbour)] = coeff

#     psi_sigma = torch.exp(log_amplitude(sigmas))
#     psi_neighbours = torch.exp(log_amplitude(neighbours))
#     signs = relsigns[sigma_idx] * relsigns[neighbours_idx]

#     diag_element = hamiltonian.apply_diag_to_basis_state(sigma)

#     return torch.sum(coeffs * psi_neighbours * signs) / psi_sigma + diag_element

In [6]:
@torch.no_grad()
def energy_loc(
    hamiltonian: ls.Operator,
    sigmas: torch.Tensor,
    log_prob_fn: Callable[[Tensor], Tensor],
    cluster: npt.NDArray[np.uint64],
    relsigns: npt.NDArray,
) -> Tensor:
    """Local energy of a spin configuration.

    Args:
        hamiltonian: Hamiltonian.
        sigma: Spin configuration.
        log_amplitude: Function that calculates logarithm of
                       the amplitude of wave function.
        cluster: Cluster of states, must contain sigma.
        relsigns: Relative signs of the cluster.

    Returns:
        Local energy.
    """
    cluster = np.sort(cluster)
    n_sigmas = len(sigmas)
    loc_energies = torch.zeros(n_sigmas)

    for i, sigma in enumerate(sigmas):
        coeffs, neighbours = zip(*hamiltonian.apply_off_diag_to_basis_state(sigma))
        sigma_idx = np.searchsorted(cluster, sigma)
        neighbours_idx = np.searchsorted(cluster, neighbours)

        neighbours = torch.from_numpy(np.array(neighbours).astype(np.int64))
        coeffs = torch.from_numpy(np.array(coeffs))
        psi_sigma = torch.exp(log_prob_fn(torch.Tensor([sigma])) / 2)
        psi_neighbours = torch.exp(log_prob_fn(neighbours) / 2)
        signs = relsigns[sigma_idx] * relsigns[neighbours_idx]

        diag_element = hamiltonian.apply_diag_to_basis_state(int(sigma.item()))

        loc_energies[i] = torch.sum(coeffs * psi_neighbours * signs) / psi_sigma + diag_element

    return loc_energies

In [7]:
def log_prob_true(states):
    return 2 * torch.log(torch.abs(torch.from_numpy(system.get_ground_state_coeffs(states))))

In [8]:
cluster = system.canonical_basis.states
relsigns = get_relsigns(system, cluster, log_prob_true)

In [9]:
energy_recovered = (
    energy_loc(
        system.hamiltonian,
        torch.from_numpy(cluster.astype(np.int64)),
        log_prob_true,
        cluster,
        relsigns,
    )
    * system.get_ground_state_coeffs(cluster) ** 2
).sum()

assert np.isclose(energy_recovered.item(), system.get_eigenstates(1)[0])

[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
/tmp/ipykernel_22215/13309741.py:39: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /home/conda/feedstock_root/build_artifacts/pytorch-recipe_1673797346477/work/aten/src/ATen/native/Copy.cpp:250.)
  loc_energies[i] = torch.sum(coeffs * psi_neighbours * signs) / psi_sigma + diag_element
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calli

In [10]:
def graph_nbd(system: SpinSystem, basis_states: npt.NDArray[np.uint64]):
    vertices = set()
    edges = set()
    for state in basis_states:
        vertices.add(state)
        for coeff, basis_state2 in system.hamiltonian.apply_off_diag_to_basis_state(state):
            vertices.add(basis_state2)
            edges.add((state, basis_state2))
    graph = nx.Graph()
    graph.add_nodes_from(vertices)
    graph.add_edges_from(edges)

    # graph = ig.Graph()
    # vertices_list = list(vertices)
    # vertice_to_id = {v: i for i, v in enumerate(vertices_list)}
    # graph.add_vertices(len(vertices), attributes={"id": vertices_list})
    # graph.add_edges([(vertice_to_id[v1], vertice_to_id[v2]) for v1, v2 in edges])
    return graph

In [11]:
# FROM: https://github.com/twesterhout/nqs-playground/blob/conda/nqs_playground/sampling.py

_SamplingOptionsBase = namedtuple(
    "_SamplingOptions",
    [
        "number_samples",
        "number_chains",
        "number_discarded",
        "sweep_size",
        "mode",
        "device",
        "other",
    ],
)


class SamplingOptions(_SamplingOptionsBase):
    r"""Options for sampling spin configurations."""

    def __new__(
        cls,
        number_samples: int,
        number_chains: int = 1,
        number_discarded: Optional[int] = None,
        sweep_size: Optional[int] = None,
        mode: Optional[str] = None,
        device: Union[str, torch.device, None] = None,
        other: Optional[Dict[str, Any]] = None,
    ):
        r"""Create SamplingOptions.

        Parameters
        ----------
        number_samples: int
            Number of samples per Markov chain. Must be a positive integer.
            'full' sampler will ignore this parameter.
        number_chains: int, optional
            Number of independent Markov chains. Must be a positive integer.
            This parameter only makes sense for MCMC samplers such as
            Metropolis-Hastings algorithm or Zanella process. Exact samplers
            ('exact' and 'autoregressive') will just multiply `number_samples`
            by `number_chains`.
        number_discarded: int, optional
            Number of samples to discard at the beginning of each Markov chain
            (i.e. how long the thermalization procedure should be). If
            specified, must be a positive integer. Otherwise, 10% of
            `number_samples` will be used. This parameter only makes sense for
            MCMC samplers (i.e. 'exact', 'autoregressive', and 'full' samplers
            will ignore this argument).
        sweep_size: int, optional
            Sweep size, i.e. how many Markov chain steps are made until the
            next sample is saved. `sweep_size = 1` means that every sample is
            saved. `sweep_size = 5` means that per every 5 steps of the MCMC
            process we only store one sample. If not specified, the default
            value of `1` will be used. This parameter only makes sense for MCMC
            samplers (i.e. 'exact', 'autoregressive', and 'full' samplers will
            ignore this argument).
        mode: str, optional
            Which algorithm to use for sampling. Valid choices are:

              * `metropolis` -- use Metropolis-Hastings algorithm with 1- or
                2-spin flips.
              * `zanella` -- use Zanella algorithm with 2-spin flips.
              * `exact` -- exactly sample from the discrete probability
                distribution using `torch.multinomial` or
                `numpy.random.choice`. This algorithm works for small systems
                only.
              * `full` -- skip sampling altogether and just return the full
                Hilbert space basis. This algorithm works for small systems
                only.
              * `autoregressive` -- assume that the probability distribution
                has a custom `sample` method and use it.
        device: str or torch.device
            On which device to run the sampling.
        other: Dict[str, Any]
            Extra arguments for a specific sampler.
        """
        number_samples = int(number_samples)
        if number_samples <= 0:
            raise ValueError("negative number_samples: {}".format(number_samples))
        number_chains = int(number_chains)
        if number_chains <= 0:
            raise ValueError("negative number_chains: {}".format(number_chains))

        if number_discarded is not None:
            number_discarded = int(number_discarded)
            if number_discarded < 0:
                raise ValueError(
                    "invalid number_discarded: {}; expected either a non-negative "
                    "integer or None".format(number_chains)
                )
        else:
            logger.info(
                "`number_discarded` not specified when constructing SamplingOptions, "
                "1/10 of `number_samples` will be used."
            )
            number_discarded = number_samples // 10
        if sweep_size is not None:
            sweep_size = int(sweep_size)
            if sweep_size <= 0:
                raise ValueError("negative sweep_size: {}".format(sweep_size))
        else:
            sweep_size = 1
            logger.warning(
                "`sweep_size` not specified when constructing SamplingOptions, "
                "`sweep_size` will be set to 1. Make sure this is what you want!"
            )
        if device is not None and not isinstance(device, torch.device):
            device = torch.device(device)
        if other is None:
            other = dict()
        return super(SamplingOptions, cls).__new__(
            cls, number_samples, number_chains, number_discarded, sweep_size, mode, device, other
        )

    def hparams(self) -> Dict[str, Any]:
        p = {
            "number_samples": self.number_samples,
            "number_chains": self.number_chains,
            "mode": self.mode,
        }
        if "mode" in ["zanella", "metropolis"]:
            p["sweep_size"] = self.sweep_size
        return p


def _determine_batch_size(options: SamplingOptions) -> int:
    batch_size = options.other.get("batch_size")
    if batch_size is None:
        batch_size = 8192
        logger.debug("'batch_size' not specified, will use the default value of 8192.")
    else:
        batch_size = int(batch_size)
        if batch_size <= 0:
            raise ValueError(
                "invalid 'batch_size': {}; expected a positive integer".format(batch_size)
            )
    return batch_size


def pad_states(states):
    return states


def split_into_batches(
    xs: Tensor | npt.NDArray | tuple[Tensor | npt.NDArray, ...] | list[Tensor | npt.NDArray],
    batch_size: int,
    device=None,
):
    batch_size = int(batch_size)
    if batch_size <= 0:
        raise ValueError("invalid batch_size: {}; expected a positive integer".format(batch_size))

    expanded = False
    if isinstance(xs, (np.ndarray, Tensor)):
        xs = (xs,)
        expanded = True
    else:
        assert isinstance(xs, (tuple, list))
    n = xs[0].shape[0]
    if any(filter(lambda x: x.shape[0] != n, xs)):
        raise ValueError("tensors 'xs' must all have the same batch dimension")
    if n == 0:
        return None

    i = 0
    while i + batch_size <= n:
        chunks = tuple(x[i : i + batch_size] for x in xs)
        if device is not None:
            chunks = tuple(chunk.to(device) for chunk in chunks)
        if expanded:
            chunks = chunks[0]
        yield chunks
        i += batch_size
    if i != n:  # Remaining part
        chunks = tuple(x[i:] for x in xs)
        if device is not None:
            chunks = tuple(chunk.to(device) for chunk in chunks)
        if expanded:
            chunks = chunks[0]
        yield chunks


def forward_with_batches(f, xs, batch_size: int, device=None) -> Tensor:
    r"""Applies ``f`` to all ``xs`` propagating no more than ``batch_size``
    samples at a time. ``xs`` is split into batches along the first dimension
    (i.e. dim=0). ``f`` must return a torch.Tensor.
    """
    if xs.shape[0] == 0:
        raise ValueError("invalid xs: {}; input should not be empty".format(xs))
    out = []
    for chunk in split_into_batches(xs, batch_size, device):
        out.append(f(chunk))
    return torch.cat(out, dim=0)


def _check_log_prob_shape(log_prob: Tensor, device: Optional[torch.device]) -> None:
    if log_prob.dim() != 1:
        raise ValueError(
            "log_prob_fn should return the logarithm of the probability, "
            "but output tensor has dimension {}; did you by accident use "
            "sign instead of amplitude network?"
            "".format(log_prob.dim())
        )
    if device is not None and log_prob.device != device:
        raise ValueError(
            "log_prob_fn should return tensors residing on {}; received "
            "tensors residing on {} instead; make sure options.device matches "
            "the location of log_prob_fn".format(device, log_prob.device)
        )

In [12]:
@torch.jit.script
def safe_exp(x: Tensor, normalise: bool = True) -> Tensor:
    r"""Calculate ``exp(x)`` avoiding overflows. Result is not equal to
    ``exp(x)``, but rather proportional to it. If ``normalise==True``, then
    this function makes sure that output tensor elements sum up to 1.
    """
    x = x - torch.max(x)
    torch.exp_(x)
    if normalise:
        x /= torch.sum(x)
    return x


In [13]:
def sample_full(
    log_prob_fn: Callable[[Tensor], Tensor], basis: ls.SpinBasis, options: SamplingOptions
) -> Tuple[Tensor, Tensor, Dict[str, Any]]:
    r"""Instead of sampling, take all basis vectors in the Hilbert space."""
    batch_size = _determine_batch_size(options)
    device = options.device
    states = torch.from_numpy(basis.states.view(np.int64))
    if device is not None:
        states = states.to(device)
    logger.debug(
        "Applying 'log_prob_fn' to all basis vectors in the Hilbert space using batch_size={}..."
        "".format(batch_size)
    )
    states = pad_states(states)
    log_prob = forward_with_batches(log_prob_fn, states, batch_size=batch_size, device=device)
    if log_prob.dim() > 1:
        log_prob.squeeze_(dim=1)
    _check_log_prob_shape(log_prob, device)
    logger.debug("Computing weights...")
    log_prob = log_prob.unsqueeze_(dim=1)
    weights = safe_exp(log_prob, normalise=True)
    states = states.unsqueeze_(dim=1)
    return states, log_prob, {"weights": weights}

In [14]:
def sample_exactly(
    log_prob_fn: Callable[[torch.Tensor], torch.Tensor], basis: ls.SpinBasis,
    options: SamplingOptions,
) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Sample states by explicitly constructing the discrete probability distribution.

    Number of samples is `options.number_chains * options.number_samples`, and
    `options.number_discarded` and `options.sweep_size` are ignored, since
    samples are already i.i.d.
    """
    states, log_prob, _extra = sample_full(log_prob_fn, basis, options)
    states = states.squeeze_(dim=1)
    log_prob = log_prob.squeeze_(dim=1)
    prob = _extra["weights"].squeeze_(dim=1)
    device = options.device
    number_samples = options.number_chains * options.number_samples
    if len(prob) < (1 << 24):
        logger.debug("Using torch.multinomial to sample indices...")
        # PyTorch only supports discrete probability distributions
        # shorter than 2²⁴.
        # NOTE: replacement=True is IMPORTANT because it more closely
        # emulates the actual Monte Carlo behaviour
        indices = torch.multinomial(prob, num_samples=number_samples, replacement=True)
    else:
        logger.debug("Using numpy.random.choice to sample indices...")
        # If we have more than 2²⁴ different probabilities chances are,
        # NumPy will complain about probabilities not being normalised
        # since float32 precision is not enough. The simplest
        # workaround is to convert the probabilities to float64 and
        # then renormalise which is what we do.
        prob = prob.to(device="cpu", dtype=torch.float64)
        prob /= torch.sum(prob)
        indices = np.random.choice(len(prob), size=number_samples, replace=True, p=prob)
        indices = torch.from_numpy(indices).to(device)

    # Choose the samples
    log_prob = log_prob[indices]
    states = states[indices]
    shape = (options.number_samples, options.number_chains)
    return states.view(*shape), log_prob.view(*shape)

In [15]:
sample_exactly(
    log_prob_true, system.basis, SamplingOptions(number_samples=10, number_chains=1, mode="exact")
)

2023-07-10 15:23:27.953 | INFO     | __main__:__new__:93 - `number_discarded` not specified when constructing SamplingOptions, 1/10 of `number_samples` will be used.
2023-07-10 15:23:27.956 | WARNING  | __main__:__new__:104 - `sweep_size` not specified when constructing SamplingOptions, `sweep_size` will be set to 1. Make sure this is what you want!
2023-07-10 15:23:27.957 | DEBUG    | __main__:_determine_batch_size:131 - 'batch_size' not specified, will use the default value of 8192.
2023-07-10 15:23:27.958 | DEBUG    | __main__:sample_full:10 - Applying 'log_prob_fn' to all basis vectors in the Hilbert space using batch_size=8192...
2023-07-10 15:23:27.962 | DEBUG    | __main__:sample_full:19 - Computing weights...
2023-07-10 15:23:28.023 | DEBUG    | __main__:sample_exactly:18 - Using torch.multinomial to sample indices...


(tensor([[682],
         [682],
         [598],
         [333],
         [173],
         [362],
         [341],
         [810],
         [613],
         [425]]),
 tensor([[-2.2051],
         [-2.2051],
         [-4.5367],
         [-4.0223],
         [-4.5367],
         [-4.5367],
         [-2.2051],
         [-4.0223],
         [-5.4819],
         [-4.5367]], dtype=torch.float64))

In [16]:
class LogProbDenseNet(nn.Module):
    def __init__(self, system: SpinSystem, n_hidden: int = 100):
        super().__init__()
        self.system = system
        self.n_hidden = n_hidden
        self.net = nn.Sequential(
            nn.Linear(system.number_spins, n_hidden), nn.ReLU(), nn.Linear(n_hidden, 1)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(
            torch.from_numpy(
                make_unpacked_configurations(x, self.system.number_spins).astype(np.float32)
            )
        )

In [17]:
log_prob_fn = LogProbDenseNet(system, n_hidden=32)

In [17]:
cluster = torch.from_numpy(system.basis.states.astype(np.int64))
energy_loc(
    system.hamiltonian,
    cluster,
    log_prob_fn,
    cluster.detach().numpy(),
    get_relsigns(system, cluster.detach().numpy(), log_prob_true),
)

[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operat

tensor([  -1.8071,  -29.8084,  -28.9654,  -29.6615,  -29.6558,   -2.1644,
         -29.8958,  -71.9693,  -73.6831,  -73.5331,  -29.9302,  -28.1953,
         -74.4118,  -70.9198,  -30.6114,  -28.1773,  -71.3834,  -29.2333,
         -29.3310,  -29.8003,   -1.9535,  -29.1153,  -73.4673,  -73.3782,
         -74.2148,  -30.1418,  -74.5576, -134.9186, -134.0625,  -73.6141,
         -73.9933, -136.7832,  -73.5730,  -76.1506,  -73.3912,  -31.1063,
         -29.0284,  -73.0624,  -72.3323,  -29.2809,  -71.3183, -134.2958,
         -72.5368,  -73.3630,  -72.5545,  -30.4991,  -29.3371,  -73.0166,
         -28.9820,  -73.3945,  -73.3857,  -30.0236,  -29.5904,  -29.1494,
         -29.9299,   -2.1060,  -29.9730,  -73.5947,  -73.1795,  -74.8835,
         -30.9701,  -72.6401, -133.5523, -133.5486,  -74.7157,  -72.4240,
        -133.3452,  -73.7005,  -73.6750,  -72.9892,  -30.6163,  -71.1987,
        -134.4191, -131.3047,  -75.7573, -130.0998, -207.2395, -132.6022,
        -132.9606, -133.2322,  -74.769

In [18]:
n_samples = 20
states, _ = sample_exactly(
    log_prob_fn,
    system.basis,
    SamplingOptions(number_samples=n_samples, number_chains=1, mode="exact"),
)
neighborhoud = graph_nbd(system, states.view(-1).detach().numpy())


2023-07-10 14:37:33.461 | INFO     | __main__:__new__:93 - `number_discarded` not specified when constructing SamplingOptions, 1/10 of `number_samples` will be used.
2023-07-10 14:37:33.462 | WARNING  | __main__:__new__:104 - `sweep_size` not specified when constructing SamplingOptions, `sweep_size` will be set to 1. Make sure this is what you want!
2023-07-10 14:37:33.463 | DEBUG    | __main__:_determine_batch_size:131 - 'batch_size' not specified, will use the default value of 8192.
2023-07-10 14:37:33.464 | DEBUG    | __main__:sample_full:10 - Applying 'log_prob_fn' to all basis vectors in the Hilbert space using batch_size=8192...
2023-07-10 14:37:33.466 | DEBUG    | __main__:sample_full:19 - Computing weights...
2023-07-10 14:37:33.468 | DEBUG    | __main__:sample_exactly:18 - Using torch.multinomial to sample indices...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_o

In [19]:
list(nx.connected_components(neighborhoud))

[{295, 535, 551, 555, 583},
 {285, 539, 541, 542, 557},
 {31,
  47,
  55,
  59,
  79,
  87,
  271,
  334,
  558,
  566,
  589,
  590,
  598,
  654,
  781,
  782,
  790},
 {93, 348, 572, 602, 604, 620, 668},
 {117,
  173,
  179,
  181,
  182,
  185,
  206,
  213,
  214,
  218,
  230,
  301,
  309,
  333,
  339,
  341,
  342,
  345,
  357,
  405,
  597,
  684,
  692,
  724,
  796,
  810,
  812,
  820,
  844,
  850,
  852,
  856,
  868,
  916},
 {677, 789, 803, 805, 806, 809, 837},
 {370, 426, 433, 434, 436, 466, 690},
 {220, 234, 236, 244, 364},
 {737, 849, 865, 866, 929},
 {391, 838, 901, 902, 906}]

In [33]:
def find_local_energies(
    system: SpinSystem,
    states: Tensor,
    neighborhoud: nx.Graph,
    log_prob_fn: Callable[[Tensor], Tensor],
) -> Tensor:
    r"""Compute local energies of given states using sign predictions 
    on the neighborhoud.

    Args:
        hamiltonian: Hamiltonian of the system.
        states: Monte-Carlo sample of states
        neighborhoud: Cluster of spins.
        log_prob_fn: Function that computes log-probabilities of states.

    Returns:
        Local energies of states.
    """
    # assert (torch.sort(states.view(-1))[0] == states.view(-1)).all(), "States must be ordered"

    energies = []
    states_order = []
    states_set = set(states.view(-1).detach().numpy())
    for cluster in nx.connected_components(neighborhoud):
        cluster: set[int]
        sigmas = torch.from_numpy(np.array(sorted(states_set & cluster)))
        cluster_np = np.array(sorted(cluster))
        relsigns = get_relsigns(system, cluster_np, log_prob_fn)
        energies.append(
            energy_loc(
                hamiltonian=system.hamiltonian,
                sigmas=sigmas,
                log_prob_fn=log_prob_fn,
                cluster=cluster_np,
                relsigns=relsigns,
            )
        )
        states_order.append(sigmas)

    energies_torch = torch.cat(energies)
    states_order_torch = torch.cat(states_order)

    sort_order = states_order_torch.argsort()

    energies_torch_sorted = energies_torch[sort_order]
    states_order_torch_sorted = states_order_torch[sort_order]

    initial_order = torch.searchsorted(states_order_torch_sorted, states.view(-1))
    return energies_torch_sorted[initial_order]

In [34]:
n_samples = 300
states, _ = sample_exactly(
    log_prob_fn,
    system.basis,
    SamplingOptions(number_samples=n_samples, number_chains=1, mode="exact"),
)
neighborhoud = graph_nbd(system, states.view(-1).detach().numpy())
assert torch.allclose(
    find_local_energies(system, states, neighborhoud, log_prob_fn),
    energy_loc(
        system.hamiltonian,
        states.view(-1),
        log_prob_fn,
        system.basis.states,
        get_relsigns(system, system.basis.states, log_prob_true),
    ),
)

2023-07-10 15:38:14.906 | INFO     | __main__:__new__:93 - `number_discarded` not specified when constructing SamplingOptions, 1/10 of `number_samples` will be used.
2023-07-10 15:38:14.908 | WARNING  | __main__:__new__:104 - `sweep_size` not specified when constructing SamplingOptions, `sweep_size` will be set to 1. Make sure this is what you want!
2023-07-10 15:38:14.910 | DEBUG    | __main__:_determine_batch_size:131 - 'batch_size' not specified, will use the default value of 8192.
2023-07-10 15:38:14.912 | DEBUG    | __main__:sample_full:10 - Applying 'log_prob_fn' to all basis vectors in the Hilbert space using batch_size=8192...
2023-07-10 15:38:14.914 | DEBUG    | __main__:sample_full:19 - Computing weights...
2023-07-10 15:38:14.915 | DEBUG    | __main__:sample_exactly:18 - Using torch.multinomial to sample indices...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
[Debug]   [LOCALE0]   Done! Returning ...
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_o

In [39]:
lattice = ChainLattice(10)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=True,
    spin_inversion=None,
    skip_symmetries_whitelist=True
)
system.hamiltonian.apply_off_diag_to_basis_state(system.basis.states[0])

2023-07-10 17:43:53.339 | WARNING  | heisenberg_hamiltonians:__init__:424 - Symmetries are not tested with ChainLattice10x1, and can produce incorrect results. Using them anyway due to skip_symmetries_whitelist=True.
2023-07-10 17:43:53.339 | DEBUG    | heisenberg_hamiltonians:__init__:456 - number_spins=10
2023-07-10 17:43:53.343 | DEBUG    | heisenberg_hamiltonians:__init__:466 - Symmetry group contains 20 elements


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-10 17:43:53.356 | DEBUG    | heisenberg_hamiltonians:__init__:475 - Hilbert space dimension is 16
[Debug]   [LOCALE0]   Calling ls_chpl_operator_apply_off_diag ...
src/BatchedOperator.chpl:248: error: halt reached - bases that require projection are not yet supported
Stacktrace

halt() at $CHPL_HOME/modules/standard/Errors.chpl:752
halt() at $CHPL_HOME/modules/standard/Errors.chpl:738
ls_chpl_operator_apply_off_diag() at src/BatchedOperator.chpl:236


: 

: 

In [65]:
log_prob_fn = LogProbDenseNet(system, n_hidden=32)
n_samples = 1024
lr = 1e-3
optimizer = torch.optim.Adam(log_prob_fn.parameters(), lr=lr)
batch_size = 64

lattice = ChainLattice(10)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=False,
    spin_inversion=None,
)
system.get_eigenstates(1)
eval_set = system.canonical_basis.states
true_amplitudes = torch.from_numpy(np.abs(system.get_ground_state_coeffs(eval_set)))

writer = SummaryWriter(
    log_dir=(
        f"experiments/2023_07_10/{datetime.now().strftime('%H_%M_%S')}"
    )
)
for step in range(100):
    states, _ = sample_exactly(
        log_prob_fn,
        system.basis,
        SamplingOptions(number_samples=n_samples, number_chains=1, mode="exact"),
    )
    states = torch.sort(states.view(-1))[0]
    neighborhoud = graph_nbd(system, states.view(-1).detach().numpy())
    E = find_local_energies(system, states, neighborhoud, log_prob_fn)

    # states = states.view(-1, states.size(-1))
    # log_probs = log_probs.view(-1)
    # weights = weights.view(-1)

    # Compute output gradient
    with torch.no_grad():
        grad = E - E.mean()
        grad *= 4
        grad = grad.view(-1, 1)
        grad_norm = torch.linalg.norm(grad)
        logger.info("‖∇E‖₂ = {}", grad_norm)
        writer.add_scalar("loss/grad", grad_norm, step)

    optimizer.zero_grad()
    # batch_size = self.config.inference_batch_size

    # Computing gradients for the amplitude network
    logger.info("Computing gradients...")
    # if _should_optimize(self.config.amplitude):
    #     self.config.amplitude.train()
    forward_fn = log_prob_fn
    for states_chunk, grad_chunk in split_into_batches((states.view(-1, 1), grad), batch_size):
        output = forward_fn(states_chunk.view(-1))
        output.backward(grad_chunk)  # , retain_graph=True)
    
    optimizer.step()
    
    predicted_amplitudes = torch.exp(log_prob_fn(eval_set) * 0.5)
    
    overlap_ = overlap(true_amplitudes, predicted_amplitudes)
    writer.add_scalar("overlap", overlap_, step)